# Gated Consensus Ensemble

Each method has a **score threshold**. A gene only gets a "vote" from a
method if its raw score passes that method's quality bar. If a method can't
find any qualifying genes for a cluster, it naturally casts zero votes.

This means different clusters can have different active method sets — e.g.
sMAPE might be the only voter for clusterThree if other methods' thresholds
filter out all their candidates.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os, pickle

%matplotlib inline

results = pickle.load(open('metrics_results.pkl', 'rb'))
constraint_patterns = results['constraint_patterns']
constraint_patternsLT = results['constraint_patternsLT']
all_methods = results['all_methods']
all_methodsLT = results['all_methodsLT']
method_names = list(all_methods.keys())

print('Methods:', method_names)

## 1. Set thresholds

Each method gets a **(direction, value)** pair:
- `('>', 0.95)` means the gene's score must be **above** 0.95 to qualify
- `('<', 0.01)` means the gene's score must be **below** 0.01 to qualify

**Tune these values** and re-run the cells below to see how the gene
selection changes. The defaults below were chosen from the score
distributions to give ~20–500 qualifying genes on clusters where the
method works, and ~0 where it doesn't.

In [ ]:
#  ============================================================
#  OPTIMIZED THRESHOLDS — from visual sweep of plots/threshold_sweep/
#  Edit these and re-run cells below to see the effect.
#  ============================================================
THRESHOLDS = {
    'Pearson':  ('>', 0.99),    # clusterTwo: 40 genes; clusterOne: 108 (shape ok, magnitude off)
    'DTW':      ('<', 0.08),    # clusterTwo: 2 genes (green); silences clusterFour
    'Cosine':   ('>', 0.999),   # Effectively silenced everywhere — cosine is not useful here
    'Frechet':  ('<', 0.03),    # clusterTwo: 3 genes (green); silences clusterFour
    'MSE':      ('<', 0.001),   # clusterOne: 25 genes (green!); clusterTwo: 3 (green!)
    'sMAPE':    ('<', 0.10),    # clusterTwo: 53 (green); clusterThree: 80 (green!) — the star
}

# How many top genes to show in the final ensemble
TOP_N = 20

print('Optimized thresholds:')
for m, (op, val) in THRESHOLDS.items():
    print(f'  {m:>8}: score {op} {val}')

In [ ]:
def get_qualifying_genes(methods_dict, cluster_name, pattern_name, thresholds):
    """For each method, return the set of genes that pass its threshold.
    
    Returns dict: method_name -> Series of scores (only qualifying genes).
    """
    qualifying = {}
    for mname, (scores, ascending) in methods_dict.items():
        if mname not in thresholds:
            continue
        op, val = thresholds[mname]
        s = scores[cluster_name][pattern_name]
        if op == '>':
            mask = s > val
        elif op == '<':
            mask = s < val
        elif op == '>=':
            mask = s >= val
        elif op == '<=':
            mask = s <= val
        qualifying[mname] = s[mask].sort_values(ascending=ascending)
    return qualifying


# Show how many genes qualify per method x cluster (LT data only)
print(f'{"=" * 60}')
print(f'LT DATA — qualifying gene counts')
print(f'{"=" * 60}')
for cluster_name in constraint_patternsLT:
    q = get_qualifying_genes(all_methodsLT, cluster_name, 'constraint', THRESHOLDS)
    print(f'\n  {cluster_name}:')
    for mname in method_names:
        n = len(q.get(mname, []))
        op, val = THRESHOLDS[mname]
        tag = ' ** SILENCED **' if n == 0 else ''
        print(f'    {mname:>8} ({op}{val}): {n:>6,} genes{tag}')

## 2. Visualize qualifying genes per method

For each cluster, plot each method's qualifying genes overlaid on the
constraint pattern. This lets you see *visually* whether a threshold is
too loose (genes everywhere) or too tight (nothing passes), and whether
the qualifying genes actually hug the pattern.

In [ ]:
def plot_qualifying_genes(methods_dict, pat_dict, thresholds, lt=False, max_plot=50):
    """For each cluster x method, plot qualifying genes overlaid on pattern."""
    x = [0, 3, 6, 9]
    suffix = 'LT' if lt else ''
    
    for cluster_name, pattern_dict in pat_dict.items():
        gene_file = f'analysis data/gene_counts{suffix}/{cluster_name}_annotated.csv'
        gene_df = pd.read_csv(gene_file, index_col=0)
        gene_df.columns = x
        
        for pattern_name, pattern_vals in pattern_dict.items():
            q = get_qualifying_genes(methods_dict, cluster_name, pattern_name, thresholds)
            active = [m for m in method_names if len(q.get(m, [])) > 0]
            silenced = [m for m in method_names if len(q.get(m, [])) == 0]
            n_active = len(active)
            if n_active == 0:
                print(f'{cluster_name}: ALL METHODS SILENCED — loosen thresholds')
                continue
            
            fig, axes = plt.subplots(1, n_active, figsize=(5 * n_active, 4.5),
                                     squeeze=False)
            
            for i, mname in enumerate(active):
                ax = axes[0, i]
                genes = q[mname].index.tolist()
                n_genes = len(genes)
                plotted = 0
                for gene in genes[:max_plot]:
                    if gene in gene_df.index:
                        ax.plot(x, gene_df.loc[gene], color='steelblue',
                                alpha=0.3, linewidth=0.8)
                        plotted += 1
                ax.plot(x, pattern_vals, color='crimson', linewidth=2.5,
                        linestyle='--')
                op, val = thresholds[mname]
                ax.set_title(f'{mname} ({op}{val})\n{n_genes:,} genes qualify',
                             fontsize=10)
                ax.set_xlabel('Timepoint')
                ax.set_xticks(x)
                if i == 0:
                    ax.set_ylabel('Gene counts')
                ax.grid(alpha=0.2)
            
            lt_tag = ' (LT)' if lt else ''
            sil_str = ', '.join(silenced) if silenced else 'none'
            fig.suptitle(f'{cluster_name}{lt_tag} — silenced: {sil_str}',
                         fontsize=12, fontweight='bold')
            plt.tight_layout()
            plt.show()

# LT data only
plot_qualifying_genes(all_methodsLT, constraint_patternsLT, THRESHOLDS, lt=True)

## 3. Gated consensus ensemble

A gene gets a vote from a method **only if it passes that method's
threshold**. Methods that produce 0 qualifying genes for a cluster are
automatically silenced — no threshold tuning needed per cluster.

In [ ]:
def gated_consensus(methods_dict, cluster_name, pattern_name, thresholds, top_n=20):
    """Score-gated ensemble. A gene gets a vote from a method only if
    it passes that method's threshold.
    
    Returns DataFrame with columns: votes, active_methods, mean_rank
    (mean rank computed only over active methods that the gene qualifies in).
    """
    q = get_qualifying_genes(methods_dict, cluster_name, pattern_name, thresholds)
    active = [m for m in method_names if len(q.get(m, [])) > 0]
    silenced = [m for m in method_names if len(q.get(m, [])) == 0]
    
    if len(active) == 0:
        return pd.DataFrame(), active, silenced
    
    # Build rank df for active methods only
    rank_df = pd.DataFrame()
    for mname in active:
        scores, ascending = methods_dict[mname]
        s = scores[cluster_name][pattern_name]
        rank_df[mname] = s.rank(ascending=ascending, method='min')
    
    # A gene gets a vote from a method only if it's in that method's qualifying set
    vote_df = pd.DataFrame(index=rank_df.index)
    for mname in active:
        qualifying_genes = set(q[mname].index)
        vote_df[mname] = rank_df.index.isin(qualifying_genes).astype(int)
    
    votes = vote_df.sum(axis=1)
    
    # Mean rank across active methods (using full ranks, not just qualifying)
    mean_rank = rank_df[active].mean(axis=1)
    
    result = pd.DataFrame({
        'votes': votes,
        'max_possible': len(active),
        'mean_rank': mean_rank,
    })
    # Only keep genes with at least 1 vote
    result = result[result['votes'] > 0]
    result = result.sort_values(['votes', 'mean_rank'], ascending=[False, True])
    
    return result.head(top_n), active, silenced


# Run the gated consensus — LT data only
print('GATED CONSENSUS ENSEMBLE (LT DATA)')
print('=' * 80)

for cluster_name in constraint_patternsLT:
    result, active, silenced = gated_consensus(
        all_methodsLT, cluster_name, 'constraint', THRESHOLDS, top_n=TOP_N
    )
    print(f'\n--- {cluster_name} ---')
    print(f'  Active:   {len(active)}/{len(method_names)} — {", ".join(active)}')
    if silenced:
        print(f'  Silenced: {", ".join(silenced)}')
    
    if result.empty:
        print('  NO QUALIFYING GENES — loosen thresholds')
        continue
    
    # Get which methods each gene qualified in
    q = get_qualifying_genes(all_methodsLT, cluster_name, 'constraint', THRESHOLDS)
    
    print(f'\n  Rank  Gene                 Votes/{len(active)}  Mean Rank  Qualified in')
    print(f'  {"-" * 72}')
    for i, (gene, row) in enumerate(result.iterrows(), 1):
        v = int(row['votes'])
        mr = row['mean_rank']
        in_methods = [m for m in active if gene in q[m].index]
        print(f'  {i:<6}{gene:<20} {v:>5}/{len(active)}  {mr:>9.1f}  {", ".join(in_methods)}')

## 4. Gated ensemble plots

In [ ]:
def plot_gated_consensus(methods_dict, pat_dict, thresholds, lt=False, top_n=20):
    x = [0, 3, 6, 9]
    suffix = 'LT' if lt else ''
    
    for cluster_name, pattern_dict in pat_dict.items():
        gene_file = f'analysis data/gene_counts{suffix}/{cluster_name}_annotated.csv'
        gene_df = pd.read_csv(gene_file, index_col=0)
        gene_df.columns = x
        
        for pattern_name, pattern_vals in pattern_dict.items():
            result, active, silenced = gated_consensus(
                methods_dict, cluster_name, pattern_name, thresholds, top_n=top_n
            )
            if result.empty:
                continue
            
            max_votes = len(active)
            fig, ax = plt.subplots(figsize=(10, 6))
            
            for gene in result.index:
                if gene in gene_df.index:
                    v = result.loc[gene, 'votes']
                    alpha = 0.2 + 0.6 * (v / max_votes)
                    ax.plot(x, gene_df.loc[gene], color='steelblue',
                            alpha=alpha, linewidth=1)
            
            ax.plot(x, pattern_vals, color='crimson', linewidth=2.5, linestyle='--')
            
            lt_tag = ' (LT)' if lt else ''
            sil_str = ', '.join(silenced) if silenced else 'none'
            title = (f'Gated Consensus | {cluster_name}{lt_tag} | '
                     f'Top {top_n} genes\n'
                     f'Active: {", ".join(active)} | Silenced: {sil_str}')
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('Timepoint')
            ax.set_ylabel('Gene counts')
            ax.set_xticks(x)
            
            legend_elements = [
                Line2D([0], [0], color='crimson', linewidth=2.5, linestyle='--',
                       label='Constraint pattern'),
                Line2D([0], [0], color='steelblue', alpha=0.8,
                       label=f'High consensus (>={max_votes-1}/{max_votes} votes)'),
                Line2D([0], [0], color='steelblue', alpha=0.3,
                       label='Low consensus (1 vote)'),
            ]
            ax.legend(handles=legend_elements, loc='upper right')
            plt.tight_layout()
            plt.show()

# LT data only
plot_gated_consensus(all_methodsLT, constraint_patternsLT, THRESHOLDS, lt=True, top_n=TOP_N)